# RQ2-v1 geometry/exposure decomposition — CPU only

This diagnostic combines frozen seed-0 functional geometry, exact structural parameter exposure, and existing seed-1/2 RQ2-v1 accuracy CSVs. It performs no training, checkpoint loading, forward pass, backward pass, or test-set recomputation.

## Secure checkout
Create Kaggle secret `github_token`. Select CPU/accelerator None.

In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

## Locate the completed RQ2-v1 output

In [ ]:
import importlib
import rq2_anchor_placement
import rq2_parameter_exposure
import rq2_geometry_exposure_decomposition
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_parameter_exposure = importlib.reload(rq2_parameter_exposure)
rq2_geometry_exposure_decomposition = importlib.reload(rq2_geometry_exposure_decomposition)
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach notebook output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(
    RQ2_INPUT, '/kaggle/working/materialized-rq2-decomposition'
)
print('Validated RQ2-v1 root:', RQ2_ROOT)

## Run decomposition
Fits use only the common holdout. The full 16-width table is still exported for anchor/high-width inspection.

In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
OUTPUT_DIR = Path('/kaggle/working/rq2-geometry-exposure-decomposition')
started = time.perf_counter()
result = rq2_geometry_exposure_decomposition.run_geometry_exposure_decomposition(
    RQ2_ROOT, OUTPUT_DIR
)
print(f'Completed in {time.perf_counter() - started:.2f} seconds')
print(json.dumps(result, indent=2))

## Inspect coefficients, per-width decomposition and proxy checks

In [ ]:
display(Markdown((OUTPUT_DIR / 'geometry_exposure_decomposition_report.md').read_text()))
display(Markdown('### Regression diagnostics'))
display(pd.read_csv(OUTPUT_DIR / 'decomposition_regression.csv'))
display(Markdown('### Full per-width table'))
display(pd.read_csv(OUTPUT_DIR / 'per_width_geometry_exposure_accuracy.csv'))
display(Markdown('### Exposure correlation with width/FLOPs proxies'))
display(pd.read_csv(OUTPUT_DIR / 'exposure_proxy_correlations.csv'))
for filename in [
    'geometry_gain_vs_accuracy.png',
    'exposure_deficit_vs_accuracy.png',
    'two_factor_fit.png',
    'exposure_by_parameter_band.png',
]:
    display(Image(filename=str(OUTPUT_DIR / filename)))

## Validate and export

In [ ]:
REQUIRED = [
    'per_width_geometry_exposure_accuracy.csv',
    'decomposition_regression.csv',
    'exposure_by_width_and_band.csv',
    'decomposition_predictions.csv',
    'exposure_proxy_correlations.csv',
    'structural_parameter_activation_bands.csv',
    'geometry_gain_vs_accuracy.png',
    'exposure_deficit_vs_accuracy.png',
    'two_factor_fit.png',
    'exposure_by_parameter_band.png',
    'geometry_exposure_decomposition_report.md',
    'geometry_exposure_decomposition_complete.json',
]
missing = [name for name in REQUIRED if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'Missing outputs: {missing}'
assert len(pd.read_csv(OUTPUT_DIR / 'per_width_geometry_exposure_accuracy.csv')) == 32
assert len(pd.read_csv(OUTPUT_DIR / 'decomposition_regression.csv')) == 6
bundle_path = Path('/kaggle/working/rq2-geometry-exposure-decomposition.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
print('Download:', bundle_path)
bundle_path